# Day4 - Part4 심화: 내 손으로 직접 만드는 오픈소스 LLM 파인튜닝 (feat. Llama 3 & QLoRA)

### 개요

앞선 파트에서는 OpenAI의 API를 통해 파인튜닝을 진행했습니다. 

이 방법은 강력한 GPU 없이도 최신 모델을 우리 목적에 맞게 튜닝할 수 있어 매우 편리합니다. 

하지만 만약, 우리가 `모델을 완전히 소유`하고 싶다면 어떨까요? 외부 API 의존성 없이 `오프라인 환경`에서 모델을 구동하고, `API 호출 비용을 전혀 내고 싶지 않다면` 어떻게 해야 할까요?

이때 필요한 것이 바로 `오픈소스 LLM의 직접 파인튜닝`입니다.

 Meta의 Llama, Mistral AI의 Mistral, Google의 Gemma 등 수많은 강력한 오픈소스 모델들은 우리가 직접 다운로드하여 자유롭게 수정하고 활용할 수 있는 길을 열어주었습니다.

하지만 여기에는 한 가지 큰 장벽이 있었습니다. 바로 `하드웨어 요구사항`입니다. 

수십억 개의 파라미터를 가진 LLM을 파인튜닝하려면 H100과 같은 고가의 전문가용 GPU가 여러 대 필요했죠.  일반 사용자의 노트북으로는 어림도 없는 일이었습니다.

이 문제를 해결하기 위해 등장한 구세주가 바로 `PEFT(Parameter-Efficient Fine-Tuning, 매개변수 효율적 파인튜닝)` 이며, 

그중에서도 가장 각광받는 기술이 `QLoRA (Quantized Low-Rank Adaptation)` 입니다. 

QLoRA는 모델의 아주 작은 일부(0.1% 미만)만 학습시키면서도 전체를 학습시킨 것과 거의 유사한 성능을 내는 마법 같은 기술입니다.  

덕분에 이제는 일반 게이밍 노트북의 GPU(VRAM 8GB 이상)로도 수십억 파라미터 모델의 파인튜닝을 시도해 볼 수 있게 되었습니다.

이번 심화 파트에서는 오픈소스 LLM 생태계의 핵심인 `Hugging Face` 라이브러리를 사용하여, Meta의 최신 모델인 `Llama 3`를 `QLoRA` 방식으로 직접 파인튜닝하는 전 과정을 다룹니다. 

API를 사용하는 것보다 조금 더 복잡하지만, 모델의 주인이 되어 내 컴퓨터에서 맞춤형 LLM을 만들어내는 값진 경험을 하게 될 것입니다.

`이번 파트의 학습 목표:`

  * 오픈소스 LLM 파인튜닝의 장단점을 상용 API 방식과 비교하여 설명할 수 있습니다.
  
  * 파인튜닝에 필요한 막대한 메모리 문제를 해결하는 `PEFT, LoRA, QLoRA`의 핵심 원리를 이해합니다.
  * `Hugging Face`의 주요 라이브러리(`transformers`, `peft`, `bitsandbytes`, `trl`)를 사용하여 파인튜닝 환경을 구축할 수 있습니다. 
  * `Llama 3.2`와 같은 최신 오픈소스 모델을 `4비트 양자화(Quantization)`를 적용하여 메모리에 로드할 수 있습니다.  (현재는 Llama 4도 출시되어 있습니다.)
  * QLoRA 설정을 구성하고, `SFTTrainer`를 사용하여 매우 간결한 코드로 파인튜닝을 실행할 수 있습니다. 
  * 학습된 어댑터(Adapter)를 저장하고, 기본 모델과 병합하여 파인튜닝된 모델의 성능을 테스트할 수 있습니다.

-----

### 1. 핵심 개념: 메모리 절약을 위한 마법, PEFT와 QLoRA

일반 노트북에서 거대 모델을 파인튜닝하기 위해, 우리는 다음 세 가지 개념의 발전 과정을 이해해야 합니다.

#### 1.1. 전체 파인튜닝 (Full Fine-Tuning)

  * `방식:` 모델의 수십억 개 파라미터 `전부`를 학습 데이터에 맞춰 업데이트합니다. 
  
  * `장점:` 모델의 성능을 최대한으로 끌어낼 수 있습니다. 
  * `단점:` 천문학적인 양의 GPU 메모리(VRAM)와 학습 시간이 필요합니다.  70억(7B) 파라미터 모델만 해도 수십 GB의 VRAM이 필요하여 일반 사용자에게는 거의 불가능합니다.

#### 1.2. LoRA (Low-Rank Adaptation)

  * `방식:` "거대한 원본 모델의 가중치는 그대로 얼려두고(freeze), 각 레이어에 아주 작은 크기의 새로운 '어댑터(Adapter)' 행렬들만 추가해서, 이 어댑터들만 학습시키자\!" 라는 아이디어입니다. 
  
  * `장점:`
      * `메모리 혁신:` 전체 파라미터의 0.1%도 안 되는 수백만 개 파라미터만 학습하므로 VRAM 사용량이 극적으로 줄어듭니다. 
      
      * `효율성:` 원본 모델은 그대로 두고 어댑터만 여러 개 만들어서, 마치 옷을 갈아입히듯 하나의 모델로 다양한 작업을 수행할 수 있습니다. (A작업용 어댑터, B작업용 어댑터 등) 
      * `성능:` 전체 파인튜닝과 거의 유사한 성능을 달성하는 것으로 알려져 있습니다. 

#### 1.3. QLoRA (Quantized Low-Rank Adaptation)

  * `방식:` LoRA를 한 단계 더 발전시킨 기술입니다. "어댑터만 학습시키는 건 좋은데, 메모리에 올려야 할 원본 모델 자체도 너무 크다. 그러니 원본 모델을 `4비트 정밀도(4-bit precision)`로 꾹 압축(양자화, Quantization)해서 메모리 사용량을 더 줄이자\!" 라는 아이디어입니다. 
  
  * `장점:`
      * `궁극의 메모리 절약:` 기존 16비트 모델을 4비트로 압축하므로, 모델을 로드하는 데 필요한 VRAM이 1/4로 줄어듭니다. 
      * `접근성:` 이 기술 덕분에, VRAM이 8GB\~16GB인 일반 게이밍 노트북에서도 70억(7B) \~ 130억(13B) 파라미터 모델의 파인튜닝이 가능해졌습니다.
  * `결론:` `QLoRA는 현재 일반 사용자가 오픈소스 LLM을 직접 파인튜닝하는 가장 현실적이고 강력한 방법`입니다. 


-----

### 2. 오픈소스 LLM 파인튜닝 실습 (Llama 3.2 & QLoRA)

이제 Hugging Face 생태계의 도구들을 활용하여 Meta의 Llama 3 8B Instruct 모델을 직접 파인튜닝 해봅시다.

#### 2.1. 환경 설정 및 모델/토크나이저 로딩

(주의: Llama 3.2 모델은 사용 전 Hugging Face에서 사용 허가 신청이 필요할 수 있습니다.)

In [1]:
!pip install transformers accelerate peft bitsandbytes datasets trl -q

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.13.0 requires numpy<=1.24.3,>=1.22, but you have numpy 1.26.4 which is incompatible.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.14.0 which is incompatible.


In [5]:
%%writefile .env
HUGGINGFACE_TOKEN=hf_xxxxxxx

Writing .env


In [1]:
# Hugging Face Hub에 로그인합니다.
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()
import os

login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [2]:
# 1. 라이브러리 임포트
import torch
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

# 2. GPU 사용 가능 여부 확인 및 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 가능한 디바이스: {device}")

# 3. 모델과 토크나이저 로딩 (CPU 환경에 맞게 수정)
model_id = "meta-llama/Llama-3.2-1B-Instruct"
# model_id = "meta-llama/Llama-3.3-70B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Llama 3는 pad_token이 없으므로, eos_token을 대신 사용합니다.
tokenizer.pad_token = tokenizer.eos_token

사용 가능한 디바이스: cpu


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

c:\Users\Admin\workspace\analysis_study\class\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--meta-llama--Llama-3.3-70B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fa

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

In [3]:
# CPU 환경에서는 양자화 없이 모델 로드
if device == "cpu":
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        device_map="auto"
    )
    print("CPU 환경에서 모델 로드 완료 (양자화 없음)")
else:
    # GPU 환경에서는 4비트 양자화 적용
    from transformers import BitsAndBytesConfig
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    print("GPU 환경에서 4비트 양자화 모델 로드 완료")

print("모델과 토크나이저 로딩 완료")

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/59.6k [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

model-00004-of-00030.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

model-00030-of-00030.safetensors:   0%|          | 0.00/2.10G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


CPU 환경에서 모델 로드 완료 (양자화 없음)
모델과 토크나이저 로딩 완료


#### 2.2. 데이터셋 준비 및 포맷팅

Llama 3.2 Instruct 모델은 특정 형식의 프롬프트를 따를 때 최고의 성능을 냅니다. 우리는 학습 데이터도 이 형식에 맞춰 가공해야 합니다.

In [4]:
# 파인튜닝을 위한 데이터 분석가 도서 관련 데이터셋 (질의응답)
qa_data = [
    {"question": "데이터 분석에서 EDA(탐색적 데이터 분석)의 목적은 무엇인가요?", "answer": "EDA는 데이터의 패턴, 이상치, 분포, 관계성을 파악하여 데이터의 특성을 이해하고, 이후 분석 방향을 결정하는 데 도움을 주는 과정입니다. 시각화와 기술통계를 통해 데이터의 품질과 특성을 종합적으로 평가합니다."},
    {"question": "파이썬 pandas에서 결측치를 처리하는 방법에는 어떤 것들이 있나요?", "answer": "결측치 처리 방법으로는 1) dropna()로 결측치가 포함된 행/열 삭제, 2) fillna()로 평균, 중앙값, 최빈값 등으로 대체, 3) forward fill/backward fill로 이전/이후 값으로 채우기, 4) 보간법(interpolation) 사용 등이 있습니다."},
    {"question": "데이터 시각화에서 히트맵(Heatmap)은 언제 사용하나요?", "answer": "히트맵은 상관관계 분석, 혼동 행렬, 시계열 데이터의 패턴 분석, 지리적 데이터 표현 등에 사용됩니다. 색상의 강도로 값의 크기를 직관적으로 표현하여 다차원 데이터의 관계를 한눈에 파악할 수 있게 해줍니다."},
    {"question": "머신러닝에서 과적합(Overfitting)을 방지하는 방법은 무엇인가요?", "answer": "과적합 방지 방법으로는 1) 교차검증(Cross-validation) 사용, 2) 정규화(Regularization) 적용, 3) 드롭아웃(Dropout) 사용, 4) 데이터 증강(Data Augmentation), 5) 앙상블 방법 사용, 6) 조기 종료(Early Stopping) 등이 있습니다."},
    {"question": "SQL에서 JOIN의 종류와 차이점을 설명해주세요.", "answer": "SQL JOIN의 주요 종류는 1) INNER JOIN: 양쪽 테이블에 모두 존재하는 데이터만, 2) LEFT JOIN: 왼쪽 테이블의 모든 데이터와 매칭되는 오른쪽 데이터, 3) RIGHT JOIN: 오른쪽 테이블의 모든 데이터와 매칭되는 왼쪽 데이터, 4) FULL OUTER JOIN: 양쪽 테이블의 모든 데이터를 포함합니다."},
    {"question": "통계에서 p-value의 의미와 해석 방법은 무엇인가요?", "answer": "p-value는 귀무가설이 참일 때 관찰된 결과보다 더 극단적인 결과가 나올 확률입니다. 일반적으로 p-value < 0.05일 때 통계적으로 유의하다고 판단하며, 이는 5% 유의수준에서 귀무가설을 기각할 수 있다는 의미입니다."},
    {"question": "데이터 전처리에서 정규화(Normalization)와 표준화(Standardization)의 차이는?", "answer": "정규화는 데이터를 0~1 범위로 변환하는 Min-Max Scaling을 의미하며, 이상치에 민감합니다. 표준화는 평균을 0, 표준편차를 1로 만드는 Z-score 변환으로, 이상치에 덜 민감하며 정규분포를 가정하는 알고리즘에 적합합니다."},
    {"question": "시계열 분석에서 계절성(Seasonality)과 추세(Trend)의 차이는?", "answer": "추세는 장기적인 증가/감소 패턴을 의미하며, 시간에 따른 일관된 방향성을 가집니다. 계절성은 일정한 주기로 반복되는 패턴으로, 월별, 계절별, 요일별 등 규칙적인 변동을 나타냅니다."},
    {"question": "A/B 테스트에서 통계적 검정력(Statistical Power)이 중요한 이유는?", "answer": "통계적 검정력은 실제로 차이가 있을 때 이를 올바르게 감지할 확률입니다. 검정력이 낮으면 실제 효과를 놓칠 위험이 있어, 적절한 표본 크기와 실험 설계가 필요합니다. 일반적으로 80% 이상의 검정력을 목표로 합니다."},
    {"question": "데이터 품질 관리에서 데이터 검증(Data Validation)의 중요성은?", "answer": "데이터 검증은 데이터의 정확성, 완전성, 일관성, 적시성을 확인하는 과정입니다. 잘못된 데이터로 인한 분석 오류를 방지하고, 신뢰할 수 있는 분석 결과를 도출하기 위해 필수적입니다."},
    {"question": "머신러닝에서 특성 선택(Feature Selection)의 방법과 장점은?", "answer": "특성 선택 방법으로는 1) 필터링(상관관계, 카이제곱 검정), 2) 래핑(순차적 선택), 3) 임베딩(Lasso, Ridge) 등이 있습니다. 장점으로는 차원 축소, 과적합 방지, 모델 해석력 향상, 계산 효율성 증대가 있습니다."},
    {"question": "데이터 시각화에서 색상 선택의 원칙은 무엇인가요?", "answer": "색상 선택 시 고려사항은 1) 색맹 친화적 색상 사용, 2) 의미에 맞는 색상 선택(빨강=위험, 초록=안전), 3) 대비가 명확한 색상 조합, 4) 일관된 색상 체계 유지, 5) 과도한 색상 사용 지양입니다."},
    {"question": "빅데이터 분석에서 분산 처리의 필요성과 방법은?", "answer": "대용량 데이터 처리를 위해 분산 처리가 필요하며, 주요 방법으로는 1) MapReduce 패러다임, 2) Spark의 인메모리 처리, 3) 분산 데이터베이스(NoSQL), 4) 클라우드 기반 처리 등이 있습니다. 병렬 처리를 통해 성능을 향상시킵니다."},
    {"question": "데이터 분석에서 이상치(Outlier) 탐지 방법은?", "answer": "이상치 탐지 방법으로는 1) IQR 방법(1.5*IQR 범위), 2) Z-score 방법(표준편차 3배), 3) DBSCAN 클러스터링, 4) Isolation Forest, 5) One-Class SVM 등이 있습니다. 도메인 지식과 함께 판단하는 것이 중요합니다."},
    {"question": "머신러닝 모델의 성능 평가 지표에는 어떤 것들이 있나요?", "answer": "주요 성능 평가 지표로는 1) 분류: 정확도, 정밀도, 재현율, F1-score, ROC-AUC, 2) 회귀: MSE, RMSE, MAE, R², 3) 클러스터링: 실루엣 계수, 칼린스키-하라바즈 지수 등이 있습니다. 문제 특성에 맞는 지표 선택이 중요합니다."},
    {"question": "데이터 분석 프로젝트의 라이프사이클은 어떻게 구성되나요?", "answer": "데이터 분석 라이프사이클은 1) 문제 정의, 2) 데이터 수집, 3) 데이터 전처리, 4) 탐색적 데이터 분석, 5) 모델링, 6) 모델 평가, 7) 결과 해석 및 시각화, 8) 인사이트 도출 및 보고서 작성의 단계로 구성됩니다."},
    {"question": "데이터 거버넌스(Data Governance)의 핵심 요소는?", "answer": "데이터 거버넌스의 핵심 요소는 1) 데이터 품질 관리, 2) 데이터 보안 및 개인정보보호, 3) 데이터 표준화, 4) 메타데이터 관리, 5) 데이터 접근 권한 관리, 6) 규정 준수, 7) 데이터 아키텍처 관리입니다."},
    {"question": "시계열 예측에서 ARIMA 모델의 구성 요소는?", "answer": "ARIMA(p,d,q) 모델은 1) AR(p): 자기회귀 모델로 과거 값들의 선형 조합, 2) I(d): 차분으로 정상성 확보, 3) MA(q): 이동평균 모델로 과거 오차들의 선형 조합으로 구성됩니다. 시계열의 추세와 계절성을 모델링합니다."},
    {"question": "파이썬에서 리스트와 튜플의 차이점은 무엇인가요?", "answer": "리스트(List)는 '가변적(mutable)'으로, 생성 후에 요소를 추가, 삭제, 변경할 수 있습니다. 반면 튜플(Tuple)은 '불변적(immutable)'으로, 한 번 생성되면 내용을 바꿀 수 없습니다."},
    {"question": "파이썬의 'if __name__ == \\'__main__\\':'은 어떤 역할을 하나요?", "answer": "이 코드는 스크립트 파일이 직접 실행될 때만 내부의 코드를 실행하고, 다른 모듈에서 임포트될 때는 실행되지 않도록 하는 역할을 합니다. 코드의 모듈화를 위해 필수적입니다."}
]
dataset = Dataset.from_list(qa_data)

# Llama 3.2 Instruct 템플릿에 맞게 데이터 포맷팅 함수 정의
def format_data(example):
    return {
        "text": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['question']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{example['answer']}<|eot_id|>"
    }

formatted_dataset = dataset.map(format_data)
print("데이터셋 포맷팅 완료")
print("포맷팅된 첫 번째 데이터 예시:\n", formatted_dataset[0]['text'])

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

데이터셋 포맷팅 완료
포맷팅된 첫 번째 데이터 예시:
 <|begin_of_text|><|start_header_id|>user<|end_header_id|>

데이터 분석에서 EDA(탐색적 데이터 분석)의 목적은 무엇인가요?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

EDA는 데이터의 패턴, 이상치, 분포, 관계성을 파악하여 데이터의 특성을 이해하고, 이후 분석 방향을 결정하는 데 도움을 주는 과정입니다. 시각화와 기술통계를 통해 데이터의 품질과 특성을 종합적으로 평가합니다.<|eot_id|>


#### 2.3. QLoRA 설정 및 Trainer 준비

이제 LoRA 어댑터를 설정하고, 모든 구성요소(모델, 데이터셋, 설정값)를 `SFTTrainer`에 전달하여 학습을 준비합니다.

In [5]:
# 1. LoRA 설정
lora_config = LoraConfig(
    r=16,                     # LoRA rank
    lora_alpha=32,            # Alpha-scaling
    lora_dropout=0.05,        # Dropout probability
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] # Llama 3의 attention layer에 적용
)

In [6]:

# 2. 학습 인자(TrainingArguments) 설정
training_args = TrainingArguments(
    output_dir="./llama3-finetuned",       # 학습 결과물 저장 경로
    per_device_train_batch_size=1,       # 메모리 부족 시 1로 설정
    gradient_accumulation_steps=4,       # 배치 크기가 작을 때 보완
    learning_rate=2e-4,                  # 학습률
    num_train_epochs=3,                  # 학습 에포크 수
    logging_steps=1,                     # 로그 출력 주기
    save_total_limit=2,                  # 체크포인트 저장 개수
    fp16=True,                           # 16비트 부동소수점 학습 활성화 (메모리 절약)
)

In [7]:
# 3. SFTTrainer (Supervised Fine-tuning Trainer) 초기화
# GPU인 경우는 주석 처리 후 사용
# GPU가 없는 환경에서 fp16을 비활성화하고 CPU 학습으로 설정하세요
training_args.fp16 = False  # fp16 비활성화
training_args.bf16 = False  # bf16도 비활성화
training_args.no_cuda = True  # CUDA 비활성화

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    peft_config=lora_config            # PEFT (LoRA) 설정
)

print("Trainer 준비 완료 (CPU 모드)")

c:\Users\Admin\workspace\analysis_study\class\.venv\Lib\site-packages\transformers\training_args.py:1577: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers and GPU quantization are unavailable.


Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

NotImplementedError: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.

#### 2.4. 학습 실행 및 모델 테스트

이제 단 한 줄의 코드로 파인튜닝을 시작하고, 학습이 끝난 모델의 성능을 테스트합니다.

In [23]:
# 1. 파인튜닝 시작
print("파인튜닝을 시작합니다...")
trainer.train()
print("파인튜닝 완료!")

# 2. 학습된 어댑터 저장
adapter_path = "../models/llm/llama3-qa-adapter"
trainer.save_model(adapter_path)
print(f"학습된 어댑터가 '{adapter_path}'에 저장되었습니다.")

파인튜닝을 시작합니다...


Step,Training Loss
1,3.038700
2,2.889200
3,2.932600
4,2.866200
5,2.635000
6,2.555100
7,2.755000
8,2.246500
9,2.298200
10,2.435800


파인튜닝 완료!
학습된 어댑터가 '../models/llm/llama3-qa-adapter'에 저장되었습니다.


In [24]:
# 3. 파인튜닝된 모델로 추론 테스트
from peft import PeftModel

# 파인튜닝된 모델 미리 로드
ft_model = PeftModel.from_pretrained(model, adapter_path)

/Users/dante/workspace/dante-code/class/star_track_python/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [25]:
def test_model_response(ft_model, tokenizer, question):
    """파인튜닝된 모델로 질문에 대한 답변을 생성하는 함수"""
    # 프롬프트 생성
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    
    # 추론 실행 - input_ids를 텐서로 변환
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"]  # 텐서로 직접 접근
    
    outputs = ft_model.generate(input_ids, max_new_tokens=200)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 답변 부분만 추출 - 안전한 방법으로 수정
    try:
        # assistant 태그로 분할 시도
        parts = response.split("<|start_header_id|>assistant<|end_header_id|>\n\n")
        if len(parts) > 1:
            answer = parts[1]
        else:
            # assistant 태그가 없으면 전체 응답 반환
            answer = response
    except:
        # 예외 발생 시 전체 응답 반환
        answer = response
    
    return answer

In [27]:
# 테스트 실행
test_question = "A/B 테스트에서 중요한 것은 무엇인가?"
answer = test_model_response(ft_model, tokenizer, test_question)

print(f"질문: {test_question}")
print(f"답변: {answer}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


질문: A/B 테스트에서 중요한 것은 무엇인가?
답변: user

A/B 테스트에서 중요한 것은 무엇인가?assistant

A/B 테스트는 데이터를 차별화하고, 이에 대한 반응을 분석하고, 그 결과에 따라 데이터를 수정하는 방법입니다. A/B 테스트의 가장 중요한 것은, 데이터의 성능을 개선하고, 가치에 대한 인식과 지지에 대한 개선이기 때문입니다.


이 과정을 통해 우리는 고가의 장비 없이도 강력한 오픈소스 LLM을 우리만의 데이터로 파인튜닝하는 데 성공했습니다. 

이처럼 오픈소스 파인튜닝은 우리에게 AI 모델을 더 깊이 이해하고 자유롭게 제어할 수 있는 강력한 무기를 제공합니다.